# Eksperimen_Teo-Prayoga-Kartika
## Proyek Akhir: Membangun Sistem Machine Learning
### Dataset: Wine Quality

**Nama Siswa:** Teo Prayoga Kartika
**Platform:** Dicoding Indonesia
**Kelas:** Membangun Sistem Machine Learning (SMSML)

---


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil diimport!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


## 2. Data Loading

Memuat dataset Wine Quality yang berisi informasi kimiawi dari sampel anggur merah beserta labelnya (tipe wine: 0, 1, 2).


In [ ]:
# Load dataset
df = pd.read_csv('../winequality_raw/winequality_raw.csv')

print(f"Dataset berhasil dimuat!")
print(f"Shape dataset: {df.shape}")
print(f"\nInformasi kolom:")
print(df.dtypes)


In [ ]:
# Tampilkan beberapa baris pertama
print("5 baris pertama dataset:")
df.head()


In [ ]:
# Tampilkan informasi dataset
print("Informasi Dataset:")
df.info()


## 3. Exploratory Data Analysis (EDA)

Pada tahap ini kita akan mengeksplorasi dataset untuk memahami distribusi data, korelasi antar fitur, dan potensi anomali.


In [ ]:
# Statistik deskriptif
print("Statistik Deskriptif Dataset:")
df.describe().round(2)


In [ ]:
# Cek missing values
print("Jumlah Missing Values per Kolom:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")


In [ ]:
# Distribusi target variable
print("Distribusi Target (Tipe Wine):")
print(df['target'].value_counts())
print(f"\nProporsi:")
print(df['target'].value_counts(normalize=True).round(3))

plt.figure(figsize=(8, 5))
target_names = ['Class 0 (Type 1)', 'Class 1 (Type 2)', 'Class 2 (Type 3)']
df['target'].value_counts().plot(kind='bar', color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black')
plt.title('Distribusi Target Variable (Tipe Wine)', fontsize=14)
plt.xlabel('Tipe Wine')
plt.ylabel('Jumlah Sampel')
plt.xticks(ticks=[0,1,2], labels=['Type 1', 'Type 2', 'Type 3'], rotation=0)
plt.tight_layout()
plt.savefig('../preprocessing/eda_target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot disimpan.")


In [ ]:
# Distribusi fitur numerik
feature_cols = [c for c in df.columns if c != 'target']
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Sembunyikan axes yang tidak terpakai
for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur Numerik', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../preprocessing/eda_feature_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot disimpan.")


In [ ]:
# Heatmap korelasi
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix,
            mask=mask,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn',
            center=0,
            vmin=-1, vmax=1,
            linewidths=0.5,
            cbar_kws={'label': 'Korelasi'})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14)
plt.tight_layout()
plt.savefig('../preprocessing/eda_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print("Heatmap disimpan.")


In [ ]:
# Deteksi Outlier menggunakan IQR
print("Deteksi Outlier (IQR Method):")
outlier_info = {}
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)][col].count()
    outlier_info[col] = outliers

outlier_df = pd.DataFrame.from_dict(outlier_info, orient='index', columns=['Jumlah Outlier'])
print(outlier_df)


In [ ]:
# Boxplot untuk visualisasi outlier
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].boxplot(df[col])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('Value')

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot Deteksi Outlier per Fitur', fontsize=14)
plt.tight_layout()
plt.savefig('../preprocessing/eda_boxplot_outliers.png', dpi=100, bbox_inches='tight')
plt.show()
print("Boxplot disimpan.")


## 4. Data Preprocessing

Tahap preprocessing meliputi:
1. Penanganan missing values (jika ada)
2. Penanganan outlier (capping/clipping)
3. Feature scaling (StandardScaler)
4. Train-Test Split


In [ ]:
# 4.1 Penanganan Missing Values
print("Penanganan Missing Values...")
df_clean = df.copy()

# Cek missing values
if df_clean.isnull().sum().sum() == 0:
    print("Tidak ada missing values. Dataset sudah bersih.")
else:
    # Isi dengan median untuk kolom numerik
    for col in feature_cols:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
    print("Missing values berhasil ditangani dengan median.")

print(f"Shape setelah penanganan missing values: {df_clean.shape}")


In [ ]:
# 4.2 Penanganan Outlier dengan IQR Capping
print("Penanganan Outlier dengan IQR Capping...")

df_no_outlier = df_clean.copy()

for col in feature_cols:
    Q1 = df_no_outlier[col].quantile(0.25)
    Q3 = df_no_outlier[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_no_outlier[col] = df_no_outlier[col].clip(lower=lower, upper=upper)

print(f"Shape setelah penanganan outlier: {df_no_outlier.shape}")
print("Outlier berhasil di-cap menggunakan metode IQR.")


In [ ]:
# 4.3 Pemisahan Fitur dan Target
X = df_no_outlier.drop('target', axis=1)
y = df_no_outlier['target']

print(f"Shape X (fitur): {X.shape}")
print(f"Shape y (target): {y.shape}")
print(f"\nDistribusi kelas target:")
print(y.value_counts())


In [ ]:
# 4.4 Feature Scaling dengan StandardScaler
print("Feature Scaling dengan StandardScaler...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Statistik sebelum scaling:")
print(X.describe().round(2).loc[['mean', 'std']].T.head(5))
print("\nStatistik setelah scaling:")
print(X_scaled.describe().round(4).loc[['mean', 'std']].T.head(5))


In [ ]:
# 4.5 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test:  {X_test.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test:  {y_test.shape}")
print(f"\nProporsi kelas pada train set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nProporsi kelas pada test set:")
print(y_test.value_counts(normalize=True).round(3))


In [ ]:
# 4.6 Simpan hasil preprocessing
import os
os.makedirs('../preprocessing/winequality_preprocessing', exist_ok=True)

# Simpan train dan test set
X_train.to_csv('../preprocessing/winequality_preprocessing/X_train.csv', index=False)
X_test.to_csv('../preprocessing/winequality_preprocessing/X_test.csv', index=False)
y_train.to_csv('../preprocessing/winequality_preprocessing/y_train.csv', index=False)
y_test.to_csv('../preprocessing/winequality_preprocessing/y_test.csv', index=False)

# Simpan dataset yang sudah diproses (gabungan)
df_preprocessed = X_scaled.copy()
df_preprocessed['target'] = y.values
df_preprocessed.to_csv('../preprocessing/winequality_preprocessing/winequality_preprocessed.csv', index=False)

print("Dataset preprocessing berhasil disimpan!")
print("File yang disimpan:")
for f in os.listdir('../preprocessing/winequality_preprocessing'):
    print(f"  - {f}")


## 5. Kesimpulan

Pada tahap eksperimen ini, telah dilakukan beberapa tahapan penting:

### Hasil EDA:
- Dataset Wine Quality memiliki **178 sampel** dengan **13 fitur** numerik dan **1 target** (tipe wine: 0, 1, 2)
- Distribusi kelas: Type 1 (59 sampel), Type 2 (71 sampel), Type 3 (48 sampel)
- Tidak ditemukan **missing values** pada dataset
- Beberapa fitur memiliki **outlier** yang telah ditangani dengan metode IQR capping
- Terdapat **korelasi tinggi** antara beberapa fitur kimia wine

### Tahapan Preprocessing:
1. ✅ **Data Loading** - Dataset berhasil dimuat
2. ✅ **EDA** - Distribusi, korelasi, dan outlier telah dianalisis
3. ✅ **Penanganan Outlier** - Menggunakan IQR capping
4. ✅ **Feature Scaling** - StandardScaler diterapkan
5. ✅ **Train-Test Split** - 80% train, 20% test dengan stratifikasi

### Output Preprocessing:
- `winequality_preprocessing/X_train.csv` - 142 sampel training
- `winequality_preprocessing/X_test.csv` - 36 sampel testing
- `winequality_preprocessing/y_train.csv` - Label training
- `winequality_preprocessing/y_test.csv` - Label testing
- `winequality_preprocessing/winequality_preprocessed.csv` - Dataset lengkap

Dataset siap digunakan untuk tahap pelatihan model.
